[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_09/listing_9.2b.ipynb)

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install gptqmodel transformers

### Listing 9.2: Quantizing a merged model to 4-bit using GPTQ via LLM Compressor

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization.gptq import GPTQModifier

# ── 1. Load the full-precision merged model ───────────────────────────────────
# AutoModelForCausalLM (not GPTQModel) is used here: llm-compressor works
# directly with standard HuggingFace models and handles quantized saving itself.
MODEL_DIR = "./merged-model"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype="auto",        # keep the model's native dtype (bfloat16 / float16)
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# ── 2. Calibration dataset ────────────────────────────────────────────────────
# GPTQ is a data-dependent algorithm: it uses a small set of representative
# samples to compute per-layer Hessians and derive optimal quantization scales.
#
# Best practice: use text that resembles your deployment distribution.
# Here we mirror the tiny inline list from the original notebook, but you can
# swap in any HuggingFace dataset, e.g.:
#   from datasets import load_dataset
#   ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:512]")
#
# Practical guidance:
#   - 128–512 samples is a common sweet spot (diminishing returns beyond 512).
#   - max_seq_length=512 covers most instruction-style prompts for a 7B model;
#     increase to 2048 if your use-case requires long context.
MAX_SEQ_LENGTH = 512
NUM_CALIBRATION_SAMPLES = 128

raw_texts = [
    "Quantization calibration example text",
    "The quick brown fox jumps over the lazy dog",
    "Large language models are trained on diverse datasets",
]

# llm-compressor expects an iterable of tokenized dicts (or a HF Dataset).
# We build a minimal Dataset to satisfy the API.
tokenized = [
    tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH)
    for text in raw_texts
]
ds = Dataset.from_list(tokenized)

# ── 3. Configure the GPTQ recipe ─────────────────────────────────────────────
# GPTQModifier is a direct conceptual replacement for gptqmodel's QuantizeConfig.
#
# Parameter explanations:
#   targets="Linear"    – quantize every nn.Linear layer in the model.
#                         You can use regex strings to restrict to specific
#                         sub-modules (e.g. "re:.*self_attn.*").
#
#   scheme="W4A16"      – shorthand for:
#                           weights  → INT4, group_size=128, symmetric
#                           activations → FP16 (unchanged / weight-only quant)
#                         This is equivalent to the original bits=4,
#                         group_size=128 in the old notebook.
#                         Alternatives:
#                           "W8A16"   – INT8 weights, FP16 activations (lossless)
#                           "W8A8"    – INT8 weights + INT8 activations (faster)
#                           "W4A16_ASYM" – asymmetric (zero-pointed) INT4
#                           "FP8"     – FP8 weights + activations (Hopper GPUs)
#
#   ignore=["lm_head"]  – skip the language-model head; quantizing it rarely
#                         helps and can harm perplexity noticeably.
#                         The original notebook left it quantized by default;
#                         this is the modern best practice.
#
# Additional knobs available on GPTQModifier (not exposed in the original):
#   dampening_frac=0.01  – numerical stabiliser for the Hessian inversion.
#                          Higher values (0.05) are safer for tricky models.
#   actorder="weight"    – reorder groups by descending weight variance before
#                          quantizing; the new default in llm-compressor 0.8,
#                          shown to recover up to 2 perplexity points vs
#                          the original gptqmodel default (desc_act=False).
#   update_size=NUM_CALIBRATION_SAMPLES – how many samples drive each update.

recipe = GPTQModifier(
    targets="Linear",
    scheme="W4A16",
    ignore=["lm_head"],
    dampening_frac=0.01,   # mirrors original damp_percent=0.05 heuristic
    actorder="weight",     # default in llm-compressor ≥0.8; better accuracy
)

# ── 4. Apply quantization ─────────────────────────────────────────────────────
# oneshot() is the single entry-point that replaces the old model.quantize()
# call.  Internally it:
#   1. Hooks into each targeted layer.
#   2. Runs the calibration dataset forward passes.
#   3. Computes Hessians and solves the GPTQ objective layer-by-layer.
#   4. Replaces weight tensors in-place with quantized INT4 + scales.
#
# pipeline="sequential" processes one transformer block at a time, which keeps
# peak VRAM at the level of a single block rather than the full model.

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

# ── 5. Save ───────────────────────────────────────────────────────────────────
# save_compressed=True writes the INT4 weights in the compressed-tensors format
# (sparse or dense packed INT4), which vLLM can load directly with no extra
# conversion.  Compare with the original which produced a gptq-format checkpoint
# readable only by libraries that understand the GPTQ packing convention.

SAVE_DIR = "./merged-model-gptq-llmcompressor"
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Quantized model saved to {SAVE_DIR}")
print("Load in vLLM with: LLM(model='./merged-model-gptq-llmcompressor')")

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


2026-05-05T20:59:04.280747+0200 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.
2026-05-05T20:59:04.476077+0200 | _make_sampler | WARNING - Requested 128 samples but the provided dataset only has 3 samples.
2026-05-05T20:59:04.476762+0200 | reset | INFO - Compression lifecycle reset
2026-05-05T20:59:04.477682+0200 | from_modifiers | INFO - Creating recipe from modifiers
2026-05-05T20:59:04.513415+0200 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-05-05T20:59:04.514236+0200 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


W0505 20:59:04.530000 585804 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/33): Calibrating: 100%|██████████| 3/3 [00:00<00:00,  5.04it/s]

2026-05-05T20:59:05.172127+0200 | compress_module_list | INFO - Quantizing model.layers.0.self_attn.q_proj using 3.0 samples
2026-05-05T20:59:05.172725+0200 | __init__ | WARNING - Could not parse CUDA_VISIBLE_DEVICES. All devices will be monitored


2026-05-05T20:59:06.509174+0200 | compress | METRIC - time 1.34s
2026-05-05T20:59:06.509821+0200 | compress | METRIC - error 0.14
2026-05-05T20:59:06.510907+0200 | _get_GPU_usage_nv | WARNING - Could not get memory info for GPU 0
2026-05-05T20:59:06.511490+0200 | compress | METRIC - GPU 0 | usage: 0.00% | total memory: 0.0 GB
2026-05-05T20:59:06.512127+0200 | compress | METRIC - Compressed module size: 33.947648 MB
2026-05-05T20:59:06.513126+0200 | compress_module_list | INFO - Quantizing model.layers.0.self_attn.k_proj using 3.0 samples
2026-05-05T20:59:07.074536+0200 | compress | METRIC - time 0.56s
2026-05-05T20:59:07.075157+0200 | compress | METRIC - error 0.07
2026-05-05T20:59:07.076046+0200 | _get_GPU_usage_nv | WARNING - Could not get memory info for GPU 0
2026-05-05T20:59:07.076339+0200 | compress | METRIC - GPU 0 | usage: 0.00% | total memory: 0.0 GB
2026-05-05T20:59:07.077003+0200 | compress | METRIC - Compressed module size: 8.486912 MB
2026-05-05T20:59:07.077566+0200 | comp

...

(33/33): Propagating: 100%|██████████| 3/3 [00:00<00:00, 6917.49it/s]

2026-05-05T21:02:51.395537+0200 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-05-05T21:02:51.395966+0200 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-05-05T21:02:51.399876+0200 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 224it [00:06, 36.45it/s]
/home/lmassaron/code/finetuning/chapter_09/.venv_ch09_llm_compressor/lib/python3.12/site-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/1 [00:00<?, ?it/s]

Quantized model saved to ./merged-model-gptq-llmcompressor
Load in vLLM with: LLM(model='./merged-model-gptq-llmcompressor')
